# Run 3 / run 4 results, locally: before vs the envelope-v2 models

**Input:** the `results/` folder of a Kaggle run (*Output* tab → `results` → download), unzipped into
`data/laya_datagen/results/run3/` or `.../run4/`; set `RUN` below. Nothing is computed on the GPU here: every number comes from the JSON files Kaggle wrote
(`eval_laya.py` already scored every model per verse on the same cards).

Sections: what ran → the verdict tables → the slices the new context is for → the boundary → the cards whose answer changed
(read them: numbers on ~30 short verses move by one card at a time).

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, Image, display

ROOT = Path.cwd().resolve()
while not (ROOT / "backend").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend" / "laya_datagen" / "kaggle"))
import eval_laya

RUN = "run4"            # which downloaded run to read: "run3" or "run4"
RES = ROOT / "data" / "laya_datagen" / "results" / RUN
DATA = ROOT / "data" / "laya_datagen" / "datasets" / "laya_v2"
assert RES.exists(), f"put the downloaded results/ folder at {RES}"
load = lambda p: json.loads(p.read_text(encoding="utf-8"))
RUNS = list(load(RES / "status.json")) if (RES / "status.json").exists() else \
    [p.stem for p in sorted(RES.glob("*.json")) if p.stem not in ("status", "rule_boundary", "before_soft_asis")
     and not p.stem.startswith(("baseline_", "train_log_"))]
before = [load(RES / "before_soft_asis.json")] if (RES / "before_soft_asis.json").exists() else []
res = {n: load(RES / f"{n}.json") for n in RUNS if (RES / f"{n}.json").exists()}
models = before + [res[n] for n in RUNS if n in res]
has = lambda m, part: any((g or {}).get(part) for g in m["groups"].values())
verdict_models = [m for m in models if has(m, "verdict")]          # a boundary-only model (run 4) has no verdict
boundary_models = [m for m in models if has(m, "boundary")]        # a verdict-only model has no boundary
teachers = [load(p) for p in sorted(RES.glob("baseline_*.json"))]
rule = load(RES / "rule_boundary.json") if (RES / "rule_boundary.json").exists() else None
print("status:", load(RES / "status.json") if (RES / "status.json").exists() else "?")
print("models:", [m["name"] for m in models])

## 1. Verdict

In [ ]:
for g in ["you", "you + Sonnet", "Sonnet · other lectures", "written text"]:
    display(Markdown(f"### {g}\n" + eval_laya.compare(verdict_models + teachers, g)))

## 2. The slices the new context is for
*short (1-3 words) with a same-surah verse around* is the الواقعة 89 / عبس 35-36 failure; *short and isolated* is where a
short match really is usually a coincidence (it should stay mostly rejected).

In [ ]:
for g in ["you", "you + Sonnet"]:
    for part in ("by context", "by span length"):
        display(Markdown(f"### {part} · {g}\n" + eval_laya.compare_slices(verdict_models, g, part)))

## 3. Boundary
At 0.5, then (run 4) with each model's thresholds tuned on its val file — what the app would use with that model.

In [ ]:
tuned = [m["tuned"] for m in boundary_models if m.get("tuned")]
for title, ms in (("at 0.5", boundary_models), ("thresholds tuned on val", tuned)):
    if not ms:
        continue
    for g in ["you", "written text", "you + Sonnet"]:
        extra = teachers if title == "at 0.5" else []
        display(Markdown(f"### {title} · {g}\n" + eval_laya.compare_boundary(([rule] if rule else []) + extra + ms, g)))
for png in ("ablation.png", "run4.png"):
    if (RES / png).exists():
        display(Image(str(RES / png)))

## 4. Training curves (val, per epoch)
Verdict accuracy and the yes/no boundary's balanced accuracy after each epoch; a val loss going up while train loss goes
down = memorising.

In [ ]:
rows = []
for n in RUNS:
    p = RES / f"train_log_{n}.json"
    if not p.exists():
        continue
    lg = load(p)
    for e in lg["epochs"]:
        rows.append({"run": n, "epoch": e["epoch"], "train loss": round(e["train_loss"], 4), "val loss": round(e["val"]["all"]["loss"], 4),
                     **{k: e["val"][k].get("balanced_accuracy", round(e["val"][k]["accuracy"], 4)) for k in e["val"] if k != "all"}})
    cal = lg.get("calibration", {})
    print(f"{n:11s} {lg['seconds'] / 60:.0f} min | {lg['train_items']} items ({lg.get('noul_true_items', 0)} 'cut off' yes) | temperatures {cal.get('temperature')}")
pd.DataFrame(rows).set_index(["run", "epoch"]) if rows else "no train logs"

## 5. The cards whose answer changed
Per verse, `before` (your app model) against one new run. `fixed` = the new model now agrees with the label, `broken` = it
no longer does. Read the broken ones first: they are what the new context costs.

In [ ]:
NEW = "both_verdict" if "both_verdict" in res else "both"     # the run to compare with `before`
v1 = {r["id"]: r for r in eval_laya.read_jsonl(DATA / "v1_test.jsonl")}
flags = load(DATA / "test_verse_flags.json")
decide = lambda p: max(eval_laya.VERDICTS, key=lambda k: (p or {}).get(k, 0.0))

def changed(new=NEW, group="you + Sonnet"):
    if not before or new not in res or not has(res[new], "verdict"):
        return pd.DataFrame()
    b, n = before[0]["verse_predictions"], res[new]["verse_predictions"]
    keep = {"you": lambda s: s == "user", "you + Sonnet": lambda s: s == "user" or s.startswith("sonnet")}[group]
    out = []
    for vid, r in v1.items():
        if vid not in b or vid not in n or not keep(r["meta"]["source"]):
            continue
        gold, pb, pn = r["meta"]["verdict"], decide(b[vid].get("verdict")), decide(n[vid].get("verdict"))
        if pb == pn:
            continue
        st = r["state"]
        out.append({"change": "fixed" if pn == gold else "broken" if pb == gold else "other", "label": gold,
                    "before": f"{pb} {b[vid]['verdict'].get(pb, 0):.2f}", NEW: f"{pn} {n[vid]['verdict'].get(pn, 0):.2f}",
                    "ref": st.get("candidate_reference"), "span": st.get("matched_span_in_lecture"),
                    "context": " ".join(str(st.get("context_before", "")).split()[-8:]) + " «…» " + " ".join(str(st.get("context_after", "")).split()[:8]),
                    **{k: v for k, v in flags.get(vid, {}).items()}})
    return pd.DataFrame(out).sort_values(["change", "label"]) if out else pd.DataFrame()

df = changed()
print(df["change"].value_counts().to_dict() if len(df) else "no changes / missing results")
pd.set_option("display.max_colwidth", 80)
df